In [1]:
#!pip install transformers torch pandas requests tabulate

In [2]:
# =========================
# 1. 导入需要的库
# =========================

import os
import requests
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from IPython.display import display, Markdown

C:\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# =========================
# 2. 自动下载 ChnSentiCorp 语料
# =========================

def download_file(url, filename):

    if os.path.exists(filename):

        print("文件已经存在：", filename)
        return

    print("正在下载：", filename)

    response = requests.get(
        url,
        timeout=60,
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    with open(filename, "wb") as f:
        f.write(response.content)

    print("下载完成：", filename)


corpus_url = (
    "https://raw.githubusercontent.com/"
    "SophonPlus/ChineseNlpCorpus/master/"
    "datasets/ChnSentiCorp_htl_all/"
    "ChnSentiCorp_htl_all.csv"
)

corpus_file = "ChnSentiCorp_htl_all.csv"


download_file(
    corpus_url,
    corpus_file
)

文件已经存在： ChnSentiCorp_htl_all.csv


In [4]:
# =========================
# 3. 读取语料
# =========================

data = pd.read_csv(
    corpus_file
)

# 删除空评论
data = data.dropna(
    subset=["review"]
)

# 重新编号
data = data.reset_index(
    drop=True
)

print(
    "语料总数：",
    len(data)
)

display(
    data.head()
)

语料总数： 7765


,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"


In [5]:
# =========================
# 4. 自动下载并加载 BERT 模型
# =========================

model_name = (
    "uer/"
    "roberta-base-finetuned-jd-binary-chinese"
)


print("正在加载BERT模型……")


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


model = AutoModelForSequenceClassification.from_pretrained(
    model_name
)


print("BERT模型加载完成！")

正在加载BERT模型……
BERT模型加载完成！


In [6]:
# =========================
# 5. 判断使用CPU还是GPU
# =========================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


model = model.to(device)

model.eval()


print(
    "当前计算设备：",
    device
)

当前计算设备： cpu


In [7]:
# =========================
# 6. BERT情感分析函数
# =========================

def bert_sentiment_analysis(text):

    # 文本编码
    inputs = tokenizer(
        str(text),

        return_tensors="pt",

        truncation=True,

        max_length=512
    )


    # 放到CPU或者GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }


    # 关闭梯度计算
    with torch.no_grad():

        outputs = model(
            **inputs
        )


    # 转换为概率
    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )


    # 找最大概率类别
    prediction = torch.argmax(
        probabilities,
        dim=1
    ).item()


    # 当前预测类别的置信度
    confidence = probabilities[
        0,
        prediction
    ].item()


    # 模型定义：
    # 0 = negative
    # 1 = positive

    if prediction == 1:

        result = "正面"

    else:

        result = "负面"


    return (
        result,
        confidence
    )

In [8]:
# =========================
# 7. 单句话测试
# =========================

text = (
    "这个酒店环境很好，"
    "服务人员也很热情，"
    "下次还会入住。"
)


result, confidence = (
    bert_sentiment_analysis(
        text
    )
)


print(
    "原文：",
    text
)

print(
    "预测情感：",
    result
)

print(
    "置信度：",
    round(confidence, 4)
)

原文： 这个酒店环境很好，服务人员也很热情，下次还会入住。
预测情感： 正面
置信度： 0.9909


In [9]:
# =========================
# 8. 对全部语料进行BERT分析
# =========================

results = []


total = len(data)


for i, row in data.iterrows():

    text = str(
        row["review"]
    )

    real_label = int(
        row["label"]
    )


    # BERT预测
    prediction, confidence = (
        bert_sentiment_analysis(
            text
        )
    )


    # 真实标签
    if real_label == 1:

        real_result = "正面"

    else:

        real_result = "负面"


    # 保存结果
    results.append({

        "编号":
            i + 1,

        "真实值":
            real_result,

        "预测值":
            prediction,

        "计算的实际值":
            round(
                confidence,
                4
            )

    })


    # 每100条显示一次进度
    if (i + 1) % 100 == 0:

        print(
            f"已经处理："
            f"{i + 1}/{total}"
        )


print(
    "\n全部分析完成！"
)

已经处理：100/7765
已经处理：200/7765
已经处理：300/7765
已经处理：400/7765
已经处理：500/7765
已经处理：600/7765
已经处理：700/7765
已经处理：800/7765
已经处理：900/7765
已经处理：1000/7765
已经处理：1100/7765
已经处理：1200/7765
已经处理：1300/7765
已经处理：1400/7765
已经处理：1500/7765
已经处理：1600/7765
已经处理：1700/7765
已经处理：1800/7765
已经处理：1900/7765
已经处理：2000/7765
已经处理：2100/7765
已经处理：2200/7765
已经处理：2300/7765
已经处理：2400/7765
已经处理：2500/7765
已经处理：2600/7765
已经处理：2700/7765
已经处理：2800/7765
已经处理：2900/7765
已经处理：3000/7765
已经处理：3100/7765
已经处理：3200/7765
已经处理：3300/7765
已经处理：3400/7765
已经处理：3500/7765
已经处理：3600/7765
已经处理：3700/7765
已经处理：3800/7765
已经处理：3900/7765
已经处理：4000/7765
已经处理：4100/7765
已经处理：4200/7765
已经处理：4300/7765
已经处理：4400/7765
已经处理：4500/7765
已经处理：4600/7765
已经处理：4700/7765
已经处理：4800/7765
已经处理：4900/7765
已经处理：5000/7765
已经处理：5100/7765
已经处理：5200/7765
已经处理：5300/7765
已经处理：5400/7765
已经处理：5500/7765
已经处理：5600/7765
已经处理：5700/7765
已经处理：5800/7765
已经处理：5900/7765
已经处理：6000/7765
已经处理：6100/7765
已经处理：6200/7765
已经处理：6300/7765
已经处理：6400/7765
已经处理：6500/7765
已经处理：6600/7765
已经处理：6700/7765
已经处理

In [10]:
# =========================
# 9. 转换成 DataFrame
# =========================

result_df = pd.DataFrame(
    results
)


print(
    "分析结果数量：",
    len(result_df)
)

分析结果数量： 7765


In [11]:
# =========================
# 10. Markdown显示前10条
# =========================

display(
    Markdown(
        "# BERT情感分析结果"
    )
)


display(
    Markdown(
        "## 前10条预测结果\n\n"
        +
        result_df
        .head(10)
        .to_markdown(
            index=False
        )
    )
)

# BERT情感分析结果

## 前10条预测结果

|   编号 | 真实值   | 预测值   |   计算的实际值 |
|-------:|:---------|:---------|---------------:|
|      1 | 正面     | 负面     |         0.5418 |
|      2 | 正面     | 正面     |         0.9937 |
|      3 | 正面     | 正面     |         0.6688 |
|      4 | 正面     | 正面     |         0.9936 |
|      5 | 正面     | 正面     |         0.7003 |
|      6 | 正面     | 正面     |         0.9294 |
|      7 | 正面     | 正面     |         0.9928 |
|      8 | 正面     | 正面     |         0.9917 |
|      9 | 正面     | 正面     |         0.9874 |
|     10 | 正面     | 正面     |         0.9876 |

In [12]:
# =========================
# 11. 整体统计
# =========================


# 判断预测是否正确
result_df["是否正确"] = (
    result_df["真实值"]
    ==
    result_df["预测值"]
)


# ---------- 基本统计 ----------

total_count = len(
    result_df
)


correct_count = (
    result_df["是否正确"]
    .sum()
)


wrong_count = (
    total_count
    -
    correct_count
)


accuracy = (
    correct_count
    /
    total_count
)


# ---------- 真实标签统计 ----------

real_positive = (
    result_df["真实值"]
    ==
    "正面"
).sum()


real_negative = (
    result_df["真实值"]
    ==
    "负面"
).sum()


# ---------- BERT预测统计 ----------

predict_positive = (
    result_df["预测值"]
    ==
    "正面"
).sum()


predict_negative = (
    result_df["预测值"]
    ==
    "负面"
).sum()


# =========================
# Markdown输出整体统计
# =========================

summary_md = f"""

# BERT情感分析整体统计


## 1. 数据基本情况

| 项目 | 数量 |
|---|---:|
| 全部语料 | {total_count} |
| 真实正面 | {real_positive} |
| 真实负面 | {real_negative} |


---


## 2. BERT预测结果

| 预测结果 | 数量 |
|---|---:|
| 正面 | {predict_positive} |
| 负面 | {predict_negative} |


---


## 3. 预测效果

| 指标 | 结果 |
|---|---:|
| 预测正确 | {correct_count} |
| 预测错误 | {wrong_count} |
| 准确率 | {accuracy:.2%} |

"""


display(
    Markdown(
        summary_md
    )
)



# BERT情感分析整体统计


## 1. 数据基本情况

| 项目 | 数量 |
|---|---:|
| 全部语料 | 7765 |
| 真实正面 | 5322 |
| 真实负面 | 2443 |


---


## 2. BERT预测结果

| 预测结果 | 数量 |
|---|---:|
| 正面 | 5077 |
| 负面 | 2688 |


---


## 3. 预测效果

| 指标 | 结果 |
|---|---:|
| 预测正确 | 6940 |
| 预测错误 | 825 |
| 准确率 | 89.38% |



In [13]:
# =========================
# 12. 保存全部结果到 CSV
# =========================


output_folder = (
    r"C:\temp"
)


# 如果文件夹不存在就自动建立
os.makedirs(
    output_folder,
    exist_ok=True
)


# 输出文件
output_file = os.path.join(
    output_folder,
    "bert_sentiment_results.csv"
)


# 只保留要求的4列
csv_result = result_df[
    [
        "编号",
        "真实值",
        "预测值",
        "计算的实际值"
    ]
]


csv_result.to_csv(
    output_file,

    index=False,

    encoding="utf-8-sig"
)


print(
    "分析完成！"
)

print(
    "CSV保存位置："
)

print(
    output_file
)

分析完成！
CSV保存位置：
C:\temp\bert_sentiment_results.csv


In [14]:
# =========================
# 10. Markdown显示部分BERT分析结果
# =========================

from IPython.display import display, Markdown


# 取前10条结果
show_n = 10

preview_rows = []

for i in range(show_n):

    text = str(data.loc[i, "review"])

    real_label = int(data.loc[i, "label"])

    prediction = result_df.loc[i, "预测值"]

    confidence = result_df.loc[i, "计算的实际值"]

    # 真实标签转换
    if real_label == 1:
        real_result = "正面"
    else:
        real_result = "负面"

    # 评论太长时截断，方便Markdown显示
    short_text = text[:40]

    if len(text) > 40:
        short_text += "..."

    preview_rows.append({
        "编号": i + 1,
        "评论内容": short_text,
        "真实值": real_result,
        "预测值": prediction,
        "置信度": confidence
    })


# 转换为DataFrame
preview_df = pd.DataFrame(preview_rows)


# 显示标题
display(
    Markdown(
        "# BERT情感分析部分结果"
    )
)


# 显示Markdown表格
display(
    Markdown(
        preview_df.to_markdown(
            index=False
        )
    )
)

# BERT情感分析部分结果

|   编号 | 评论内容                                                                            | 真实值   | 预测值   |   置信度 |
|-------:|:------------------------------------------------------------------------------------|:---------|:---------|---------:|
|      1 | 距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的...       | 正面     | 负面     |   0.5418 |
|      2 | 商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!                               | 正面     | 正面     |   0.9937 |
|      3 | 早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很... | 正面     | 正面     |   0.6688 |
|      4 | 宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很...  | 正面     | 正面     |   0.9936 |
|      5 | CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风                   | 正面     | 正面     |   0.7003 |
|      6 | 总的来说，这样的酒店配这样的价格还算可以，希望他赶快装修，给我的客人留些好的印象    | 正面     | 正面     |   0.9294 |
|      7 | 价格比比较不错的酒店。这次免费升级了，感谢前台服务员。房子还好，地毯是新的，比上... | 正面     | 正面     |   0.9928 |
|      8 | 不错，在同等档次酒店中应该是值得推荐的！                                            | 正面     | 正面     |   0.9917 |
|      9 | 入住丽晶，感觉很好。因为是新酒店，的确有淡淡的油漆味，房间内较新。房间大小合适，... | 正面     | 正面     |   0.9874 |
|     10 | 1。酒店比较新，装潢和设施还不错，只是房间有些油漆味。2。早餐还可以，只是品种不...   | 正面     | 正面     |   0.9876 |

In [15]:
# =========================
# 11. Markdown显示预测错误案例
# =========================

wrong_df = result_df[
    result_df["真实值"]
    !=
    result_df["预测值"]
].head(5)


wrong_rows = []


for index, row in wrong_df.iterrows():

    text = str(
        data.loc[index, "review"]
    )

    short_text = text[:50]

    if len(text) > 50:
        short_text += "..."

    wrong_rows.append({
        "编号": row["编号"],
        "评论内容": short_text,
        "真实值": row["真实值"],
        "预测值": row["预测值"],
        "置信度": row["计算的实际值"]
    })


wrong_preview = pd.DataFrame(
    wrong_rows
)


display(
    Markdown(
        "## BERT预测错误案例"
    )
)


display(
    Markdown(
        wrong_preview.to_markdown(
            index=False
        )
    )
)

## BERT预测错误案例

|   编号 | 评论内容                                                                                              | 真实值   | 预测值   |   置信度 |
|-------:|:------------------------------------------------------------------------------------------------------|:---------|:---------|---------:|
|      1 | 距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.          | 正面     | 负面     |   0.5418 |
|     20 | 价格偏高,好象连云港这地方的酒店都偏贵.早饭不好.房间还不错,窗外风景还行.最重要是房间的窗户很隔音...    | 正面     | 负面     |   0.6553 |
|     21 | 知道网线接口在哪儿吗？比高家庄的地道口还隐蔽。在床头柜后面！想不道吧？看你怎么用。1：自带4米以上网... | 正面     | 负面     |   0.905  |
|     24 | 总体还可以,就是前台服务员还不够敬业,在登记入住后,看都没看是不是本人,就把我客人的护照给了其它人,...    | 正面     | 负面     |   0.5851 |
|     27 | 房间设备太破,连喷头都是不好用,空调几乎感觉不到,虽然我开了最大另外就是设备维修不及时,洗澡用品感觉...   | 正面     | 负面     |   0.6613 |